CRIANDO AS BASES

In [1]:
set.seed(2026); n <- 200                                        # sorteio reproduzivel; 200 anuncios
anuncios <- data.frame(dist  = round(runif(n, 0.1, 5), 1),      # distancia ate a praia (km)
                       fotos = sample(2:20, n, replace = TRUE))  # fotos no anuncio, de 2 a 20
lambda <- exp(1.6 - 0.5 * anuncios$dist + 0.06 * anuncios$fotos)  # a verdade: visitas esperadas em 30 dias
anuncios$visitas <- rpois(n, lambda)                            # visitas observadas em 30 dias
set.seed(11)                                                    # outro sorteio, para o Modulo 5
anuncios$visitas_od <- rpois(n, lambda * rgamma(n, shape = 2, rate = 2))  # contagens mais espalhadas
set.seed(7)                                                     # a base do Modulo 4
an2 <- data.frame(dist  = round(runif(n, 0.1, 5), 1),           # outros 200 anuncios, mesmas colunas
                  fotos = sample(2:20, n, replace = TRUE))       # fotos no anuncio
an2$dias <- round(runif(n, 5, 15) + 9 * an2$dist)               # dias no ar: perto da praia vende antes
an2$visitas <- rpois(n, an2$dias * exp(-1.8 - 0.5 * an2$dist + 0.06 * an2$fotos))  # visitas no periodo
c(sum(anuncios$visitas), sum(anuncios$visitas_od), sum(an2$visitas))  # confira: 836 775 550

[1] 836 775 550

POR QUE A RETA NÃO SERVE PARA CONTAGENS

In [2]:
d8 <- data.frame(dist    = c(0.2, 0.5, 1, 1.5, 2, 3, 4, 4.5),  # distancia ate a praia (km)
                 visitas = c(9, 7, 5, 4, 3, 1, 1, 0))           # visitas no mes
reta <- lm(visitas ~ dist, data = d8)                           # o MQO da Aula 4
coef(reta)                                                      # 7,63 e -1,86
predict(reta, data.frame(dist = c(4.5, 5)))                     # -0,73 e -1,66 visitas

(Intercept)        dist 
   7.630133   -1.858746

1          2 
-0.7342258 -1.6635990

AJUSTAR E LER NO R

In [3]:
m <- glm(visitas ~ dist + fotos, family = poisson, data = anuncios)  # regressao de Poisson
round(coef(m), 4)                    # escala do log: 1,5738  -0,5217  0,0653
round(exp(coef(m)), 4)               # multiplicadores: 4,8248  0,5935  1,0674
novo <- data.frame(dist = 1, fotos = 12)  # um anuncio a 1 km, com 12 fotos
predict(m, novo)                          # 1,835: o LOG das visitas esperadas
predict(m, novo, type = "response")       # 6,27 visitas em 30 dias (a verdade: 6,17)

(Intercept)        dist       fotos 
     1.5738     -0.5217      0.0653

(Intercept)        dist       fotos 
     4.8248      0.5935      1.0674

1 
1.835214

1 
6.266475

O OFFSET

In [4]:
sem <- glm(visitas ~ dist + fotos, family = poisson, data = an2)  # ignora os dias no ar
com <- glm(visitas ~ dist + fotos + offset(log(dias)),            # modela visitas por dia
           family = poisson, data = an2)                          # mesma familia
round(exp(coef(sem)), 3)    # dist: 0,832, a praia parece valer pouco
round(exp(coef(com)), 3)    # dist: 0,621, perto da verdade, 0,607
predict(com, data.frame(dist = 1, fotos = 12, dias = 20), type = "response")  # 3,85 visitas

(Intercept)        dist       fotos 
      2.304       0.832       1.058

(Intercept)        dist       fotos 
      0.160       0.621       1.057

1 
3.849521

SOBREDISPERSÃO

In [5]:
mp <- glm(visitas_od ~ dist + fotos, family = poisson, data = anuncios)       # Poisson nas contagens espalhadas
sum(residuals(mp, type = "pearson")^2) / df.residual(mp)                     # phi: 3,37 (a Poisson pede 1)
mq <- glm(visitas_od ~ dist + fotos, family = quasipoisson, data = anuncios)  # mesma conta, erro corrigido
round(cbind(coef(mp), coef(mq)), 4)                                          # coeficientes iguais
round(cbind(summary(mp)$coefficients[, 2],                                   # erros padrao: Poisson
            summary(mq)$coefficients[, 2]), 4)                               # e quase-Poisson: x 1,84

[1] 3.371795

(Intercept),1.4452,1.4452
dist,-0.4444,-0.4444
fotos,0.0609,0.0609


(Intercept),0.1112,0.2041
dist,0.0303,0.0556
fotos,0.0067,0.0124


POISSON CONTRA MQO

In [6]:
set.seed(1); dobra <- sample(rep(1:5, length = n))              # as 5 dobras dos laboratorios
rmse_cv <- function(ajusta) sqrt(mean(sapply(1:5, function(j) {  # RMSE da validacao cruzada
  tr <- anuncios[dobra != j, ]; va <- anuncios[dobra == j, ]       # treino e validacao da dobra j
  mean((va$visitas - ajusta(tr, va))^2) })))                       # erro quadratico medio da dobra
rmse_cv(function(tr, va) rep(mean(tr$visitas), nrow(va)))          # linha de base: 3,99
rmse_cv(function(tr, va) predict(lm(visitas ~ dist + fotos, data = tr), va))  # MQO: 2,39
rmse_cv(function(tr, va) predict(glm(visitas ~ dist + fotos, family = poisson,  # Poisson:
                                     data = tr), va, type = "response"))        # 1,76

[1] 3.990111

[1] 2.386974

[1] 1.757921

PARTE D, MÓDULO 0: LENDO O BANCO:

In [10]:
dados <- read.csv("/content/exportacao_cargas.csv.gz",    # o banco do trabalho (solto em /content)
                  stringsAsFactors = FALSE, check.names = FALSE)
str(dados)                                                # colunas e tipos
nrow(dados)                                               # o n, para conferir com a ficha

'data.frame':	1097014 obs. of  16 variables:
 $ TOTAL_TEU      : int  92 28 598 30 317 14 12 2 53 15 ...
 $ NUMERO_VIAGEM  : int  4659 3888 4731 4646 5010 4856 5006 4790 4237 4465 ...
 $ TIPO_NAVEGACAO : chr  "LONGO CURSO" "LONGO CURSO" "CABOTAGEM" "LONGO CURSO" ...
 $ MOVIMENTO      : chr  "CONVENCIONAL" "REMOÇÃO" "CONVENCIONAL" "CONVENCIONAL" ...
 $ ANO            : int  2012 2012 2012 2012 2012 2012 2012 2012 2012 2012 ...
 $ TERMINAIS      : chr  "SANTOS BRASIL" "SANTOS BRASIL" "SANTOS BRASIL" "SANTOS BRASIL" ...
 $ TOTAL_TONELADAS: chr  "1504,855" "190,55" "5631,35" "596,37" ...
 $ NATUREZA_CARGA : chr  "CARGA CONTEINERIZADA" "CARGA CONTEINERIZADA" "CARGA CONTEINERIZADA" "CARGA CONTEINERIZADA" ...
 $ TOTAL_UNID     : int  76 14 335 24 227 14 12 1 53 10 ...
 $ MES            : int  11 10 10 11 11 11 11 11 10 10 ...
 $ CLASSENAVIO    : chr  "PORTA-CONTAINERS" "PORTA-CONTAINERS" "PORTA-CONTAINERS" "PORTA-CONTAINERS" ...
 $ SENTIDO        : chr  "DESEMBARQUE" "EMBARQUE" "DESEMBARQUE" 

[1] 1097014

PARTE D, MÓDULO 7: A CONTAGEM:

In [11]:
table(dados$MES)                                          # confere se MES vem como 1 a 12
conteiner <- subset(dados, NATUREZA_CARGA == "CARGA CONTEINERIZADA" &  # so carga conteinerizada
                           as.numeric(ANO) <= 2025)                    # 2026 e um ano incompleto
conteiner$MES <- as.integer(conteiner$MES)                # mes como numero
stopifnot(!anyNA(conteiner$MES))                          # se falhar, o MES nao veio como numero

tot <- table(conteiner$TERMINAIS)                         # registros por terminal, no periodo todo
raros <- names(tot)[tot < 200]                            # terminais quase sem registro
length(raros)                                             # quantos terminais serao juntados
conteiner$TERMINAIS[conteiner$TERMINAIS %in% raros] <- "OUTROS"  # junta os raros

cont <- as.data.frame(table(conteiner$TERMINAIS, conteiner$ANO, conteiner$MES))  # uma linha por terminal x ano x mes
names(cont) <- c("terminal", "ano", "mes", "n")           # n: a contagem de registros
cont$ano <- as.numeric(as.character(cont$ano))            # ano como numero (tendencia)
mes_num <- as.integer(as.character(cont$mes))             # mes como numero, para achar os dias do mes
bissexto <- cont$ano %% 4 == 0 & (cont$ano %% 100 != 0 | cont$ano %% 400 == 0)   # ano bissexto?
cont$dias <- c(31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31)[mes_num] +        # dias do mes: a exposicao
             (mes_num == 2 & bissexto)                    # fevereiro de ano bissexto tem 29
nrow(cont)                                                # quantas linhas tem a nova tabela
sum(cont$n == 0)                                          # quantas combinacoes deram zero
summary(cont$n)                                           # media e espalhamento da contagem


    1     2     3     4     5     6     7     8     9    10    11    12 
88153 83488 92992 90860 95523 93906 91576 93520 92627 95435 88958 89976 

[1] 9

[1] 5040

[1] 2070

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    0.0     0.0     3.0   188.6    41.0  2225.0 

PARTE D, MÓDULO 8: AJUSTAR, LER E CONFERIR:

In [12]:
m <- glm(n ~ terminal + ano + mes + offset(log(dias)),    # regressao de Poisson com offset
         family = poisson, data = cont)
round(exp(coef(m)), 3)                                    # razoes de taxas: e^beta
exp(coef(m)["ano"])                                       # multiplicador da taxa por ano a mais
phi <- sum(residuals(m, type = "pearson")^2) / df.residual(m)  # phi: perto de 1?
phi                                                       # se passar de 1,5, usar a quase-Poisson
summary(m)$coefficients                                   # estimativas e erros padrao

mq <- glm(n ~ terminal + ano + mes + offset(log(dias)),   # mesma conta, com erro padrao corrigido
          family = quasipoisson, data = cont)
round(cbind(poisson = summary(m)$coefficients["ano", 2],  # erro padrao do ano na Poisson
            quase   = summary(mq)$coefficients["ano", 2]), 5)  # e na quase-Poisson

(Intercept)                 terminalBTP 
                      0.000                      62.289 
                terminalCBA             terminalDEICMAR 
                      0.156                       1.619 
terminalDPWORLD (EMBRAPORT)            terminalECOPORTO 
                     38.112                      20.337 
           terminalLIBRA 35            terminalLIBRA 37 
                     37.653                       0.507 
          terminalLOCALFRIO             terminalMARIMEX 
                      1.865                       1.520 
           terminalMESQUITA                 terminalNST 
                      0.402                       0.085 
             terminalOutros              terminalOUTROS 
                      4.234                       0.135 
           terminalRODRIMAR       terminalSANTOS BRASIL 
                     11.546                     130.945 
           terminalTERMARES                 terminalTEV 
                      0.980                       0.265 
         terminalTRANSBRASA            terminalUSIMINAS 
                      1.042                       0.202 
                        ano                        mes2 
                      1.012                       1.033 
                       mes3                        mes4 
                      1.040                       1.052 
                       mes5                        mes6 
                      1.072                       1.086 
                       mes7                        mes8 
                      1.085                       1.107 
                       mes9                       mes10 
                      1.136                       1.134 
                      mes11                       mes12 
                      1.091                       1.072

ano 
1.012187

[1] 150.4463

,Estimate,Std. Error,z value,Pr(>|z|)
(Intercept),-25.41637922,0.3424328744,-74.2229532,0.000000e+00
terminalBTP,4.13177692,0.0183453883,225.2215572,0.000000e+00
terminalCBA,-1.85570194,0.0494962582,-37.4917622,1.254683e-307
terminalDEICMAR,0.48206213,0.0231468424,20.8262589,2.502627e-96
terminalDPWORLD (EMBRAPORT),3.64052808,0.0184370992,197.4566631,0.000000e+00
terminalECOPORTO,3.01241904,0.0186419745,161.5933457,0.000000e+00
terminalLIBRA 35,3.62840025,0.0184399748,196.7681781,0.000000e+00
terminalLIBRA 37,-0.67835158,0.0313684080,-21.6253110,1.038154e-103
terminalLOCALFRIO,0.62336140,0.0225571243,27.6347901,4.251674e-168
terminalMARIMEX,0.41895437,0.0234328921,17.8789017,1.721959e-71


poisson,quase
0.00017,0.00208


PARTE D, MÓDULO 9: VALIDAR E DECIDIR:

In [13]:
set.seed(1); dobra <- sample(rep(1:5, length = nrow(cont)))      # as 5 dobras de sempre
rmse_cv <- function(ajusta) sqrt(mean(sapply(1:5, function(j) {  # RMSE da validacao cruzada
  tr <- cont[dobra != j, ]; va <- cont[dobra == j, ]               # treino e validacao da dobra j
  mean((va$n - ajusta(tr, va))^2) })))                             # erro quadratico medio da dobra
rmse_cv(function(tr, va) rep(mean(tr$n), nrow(va)))                # linha de base
rmse_cv(function(tr, va) predict(lm(n ~ terminal + ano + mes, data = tr), va))  # MQO
rmse_cv(function(tr, va) predict(glm(n ~ terminal + ano + mes + offset(log(dias)),  # Poisson
                                     family = poisson, data = tr), va, type = "response"))  # na escala da contagem

sum(unlist(lapply(1:5, function(j) {                       # previsoes negativas do MQO na validacao
  tr <- cont[dobra != j, ]; va <- cont[dobra == j, ]
  predict(lm(n ~ terminal + ano + mes, data = tr), va) < 0 })))

[1] 433.2219

[1] 219.2098

[1] 212.8077

[1] 1067